# L14 · Real Small-Model Training and Server Scaling

## Goal

- read a download guard
- interpret a LoRA memory estimate
- separate laptop and server boundaries

## Setup

This cell fixes CPU, seed, offline status, and the split hash first. Toy code uses deterministic CPU operations; package trainers retain their strict global default.

In [1]:
import hashlib, json, os, platform, random, sys
from pathlib import Path
os.environ.setdefault("TORCH_DEVICE_BACKEND_AUTOLOAD", "0")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the RL-study repository")
sys.path.insert(0, str(ROOT / "src"))
import torch
from rl_study import __version__
from rl_study.data import build_tiny_reasoning
from rl_study.runtime import resolve_device, seed_everything
# These notebooks use only deterministic CPU toy kernels.  PyTorch 2.13's global
# guard imports the full Inductor stack, so keep the package's strict default for
# trainers while avoiding that unrelated startup cost in fresh teaching kernels.
seed_everything(42, deterministic=False)
random.seed(42)
language = os.environ.get("RL_STUDY_NOTEBOOK_LANGUAGE", "ko")
resolution = resolve_device("cpu")
dataset = build_tiny_reasoning(seed=42)
config_hash = "sha256:" + hashlib.sha256(b"L14:toy:42").hexdigest()
print(f"lesson=L14 language={language} profile=toy")
print("seed=42 network_required=False deterministic_scope=seeded_cpu_toy")
print(f"python={platform.python_version()} rl_study={__version__} torch={torch.__version__}")
print(f"requested_device=cpu resolved_device={resolution.resolved} fallback_used={resolution.fallback_used}")
print(f"config_hash={config_hash} data_split_hash={dataset.split_hash}")

lesson=L14 language=en profile=toy
seed=42 network_required=False deterministic_scope=seeded_cpu_toy
python=3.10.12 rl_study=0.1.0.dev0 torch=2.13.0
requested_device=cpu resolved_device=cpu fallback_used=False
config_hash=sha256:266a5466f8136e9801321a6a19b4caba3ccc1bb62cf9958ffbdc632d74e61de4 data_split_hash=sha256:f238657bbf6c0a112debf7ef3ffafb452c14308dfb5ce57d9abe4f77ac1deedd


## Steps

### 1. Position and core equation

⏱ 5 min · 1/3 section · [CORE]

Position: toy algorithms → **real-model profiles** → server recipes

$$M_{train}\approx M_{weights}+M_{gradients}+M_{optimizer}+M_{activations}+M_{headroom}$$

Toy and public-model paths share algorithm APIs, but downloads, tokenizer revisions, dtype, adapters, and devices add failure boundaries. The laptop preset uses LoRA with conservative headroom; server presets isolate distributed-framework responsibilities behind adapters.

### 2. Run with small numbers

⏱ 6 min · 2/3 section · [CORE]

**Predict first:** If the model is uncached and no approval flag is set, what must stop first: optional import or download? Write an answer for 20 seconds, then run the cell.

<details><summary>Show answer</summary>The download guard must stop before optional framework imports so no environment mutation or large transfer begins.</details>

In [2]:
from rl_study.adapters import MODEL_PRESETS, enforce_download_guard, estimate_training_memory
from rl_study.errors import DownloadApprovalRequired
manifest = MODEL_PRESETS["laptop-smoke"]
memory = estimate_training_memory(
    manifest, adapter="lora", dtype="float32", batch_size=1, sequence_length=128
)
guard_blocked = False
try:
    enforce_download_guard(manifest, cached=False, accept_download=False)
except DownloadApprovalRequired:
    guard_blocked = True
print({"model": manifest.hub_id, "revision": manifest.revision[:12],
       "expected_download_mb": round(manifest.expected_bytes / 1e6, 1),
       "recommended_memory_gib": round(memory.recommended_bytes / 2**30, 2),
       "guard_blocked_before_download": guard_blocked})

{'model': 'HuggingFaceTB/SmolLM2-135M-Instruct', 'revision': '12fd25f77366', 'expected_download_mb': 269.1, 'recommended_memory_gib': 1.59, 'guard_blocked_before_download': True}


### 3. Implementation anatomy

⏱ 6 min · 3/3 section · [DEEP DIVE]

**Why this implementation:** Exact peak memory depends on hardware and kernels, so estimates stay separate from measurements. QLoRA lowers memory but adds quantization-backend compatibility boundaries.

**Common trap:** Pinning only `model_id` allows the same config to fetch different weights. Store model/tokenizer revisions and expected bytes together. Regression tests: `test_large_download_guard_runs_before_optional_import`.

**Checkpoint:** Continue when you can explain just one printed value.

## Checks

In [3]:
assert guard_blocked and manifest.expected_bytes > 100_000_000
print("checks=passed")

checks=passed


**Recall:** Which three conditions must preflight recheck even when the memory estimate passes? Answer in one or two sentences.

## Mistakes I Revisit

- Assuming a finite loss proves the implementation is correct.
- Merging `terminated` with `truncated`, or prompt with action.
- Turning one tiny seed into an algorithm ranking.

## 60-Second Recap

- **Run conclusion:** The laptop smoke preset reports about 269.1MB download and 1.59GiB recommended memory, and blocks an unapproved transfer before download.
- Executable checks: `test_large_download_guard_runs_before_optional_import`.
- The output is a fixed-seed toy run, not a paper-scale result.

## Next Steps

1. L15 moves beyond single responses to trajectories with tool calls and multiple observations.
2. Break one `[CORE]` assertion and read the failure.
3. Open the package test and connect the notebook equation to its production guard.

## Sources

- `framework-trl` — `docs/sources.yml`
- `framework-verl` — `docs/sources.yml`
- `framework-openrlhf` — `docs/sources.yml`